# Two curricula with the anchor loss removed

Select a **GPU runtime**, then **Run all**. Both recipes retrain their initial adapters **from identity** on the saved successful near-band experience, then continue through the same three downward bands. **Every fit uses ranking loss only; anchor_weight=0.** Old anchored adapter weights are not used to initialize the new fits.

The original recipe uses temporal pairs; clean additionally requires at least 0.5 actual goal progress. Geometry, cases, shared saved goal, architecture and fitting budgets remain the same. The goal-reference inconsistency is deliberately left unchanged for this experiment.

Outputs go to **multiband_no_anchor_v1**. Previous experiments remain read-only. The full run schedules at most **840 planning/collection episodes**, plus two initial fits and up to six later fits of 500 optimizer updates each. Progress resumes from validated completed artifacts.


In [ ]:
from google.colab import drive
from pathlib import Path
import json, shutil, signal, subprocess, sys, time
import torch

drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Select a GPU runtime before continuing.'
DRIVE = Path('/content/drive/MyDrive')
REPO_URL = 'https://github.com/kmylonas/i-jepa-wm.git'
SOURCE_REF = 'curriculum-cost-adapter'
SOURCE_COMMIT = None
REPO = Path('/content/i-jepa-wm-no-anchor')
DINO_ROOT = Path('/content/dino_wm')
PARENT = DRIVE / 'datasets/curriculum_cost_adapter/pilot_v1'
CLEAN = DRIVE / 'datasets/curriculum_cost_adapter/clean_pair_v1'
ADJACENT = DRIVE / 'datasets/curriculum_cost_adapter/adjacent_band_v1'
OUTPUT = DRIVE / 'datasets/curriculum_cost_adapter/multiband_no_anchor_v1'
BASELINE = DRIVE / 'datasets/curriculum_cost_adapter/multiband_curriculum_v1'
ENCODER_DRIVE = DRIVE / 'checkpoints/IN1K-vit.h.14-300e.encoder-only.pth.tar'
WM_DRIVE = DRIVE / 'datasets/wall_matched_fixed_multi_v3/fixed/world_model/best.pt'
ARTIFACTS = Path('/content/curriculum-no-anchor-artifacts')
DINO_COMMIT = '0a9492fa12044b852ae9e001cc74604b79c8bb0c'

def shell(command, *, cwd=None, log=None):
    command = [str(arg) for arg in command]
    print('$', ' '.join(command), flush=True)
    stream = open(log, 'a', buffering=1) if log else None
    try:
        with subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
            try:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    if stream: stream.write(line)
                returncode = process.wait()
            except BaseException:
                process.send_signal(signal.SIGINT)
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired: process.terminate()
                raise
        if returncode: raise subprocess.CalledProcessError(returncode, command)
    finally:
        if stream: stream.close()


## Frozen models and source artifacts

Use the same Drive folder layout as before. This experiment reuses the saved near-band experience, case definitions and target. It creates fresh identity-initialized adapters; the first fit trains those adapters before their initial evaluations. Later band fits start from that recipe's latest checkpoint with fresh Adam moments.


In [ ]:
for source in (PARENT, CLEAN, ADJACENT, BASELINE):
    assert source.is_dir(), f'Missing completed experiment: {source}'
    assert OUTPUT.resolve() != source.resolve() and not OUTPUT.resolve().is_relative_to(source.resolve()), 'Output must be outside both sources'
assert (CLEAN / 'run_0/goal_tokens.pt').is_file(), 'Run the clean-pair notebook first'
if not REPO.exists():
    shell(['git', 'clone', '--branch', SOURCE_REF, '--single-branch', REPO_URL, REPO])
else:
    assert (REPO / '.git').exists(), f'{REPO} is not a Git checkout'
    assert subprocess.check_output(['git', '-C', str(REPO), 'remote', 'get-url', 'origin'], text=True).strip() == REPO_URL
    assert not subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain'], text=True).strip(), 'Source clone has local changes'
    shell(['git', '-C', REPO, 'fetch', 'origin', f'refs/heads/{SOURCE_REF}:refs/remotes/origin/{SOURCE_REF}'])
shell(['git', '-C', REPO, 'checkout', '--detach', SOURCE_COMMIT or f'origin/{SOURCE_REF}'])
source_commit = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
print('Multi-band source revision:', source_commit)

shell([sys.executable, '-m', 'pip', 'install', '-q', 'gym==0.26.2', 'omegaconf', 'pyyaml', 'scipy', 'einops', 'matplotlib', 'imageio', 'imageio-ffmpeg', 'pillow'])
if not DINO_ROOT.exists(): shell(['git', 'clone', 'https://github.com/gaoyuezhou/dino_wm.git', DINO_ROOT])
assert not subprocess.check_output(['git', '-C', str(DINO_ROOT), 'status', '--porcelain', '--untracked-files=no'], text=True).strip(), 'DINO-WM source has local changes'
shell(['git', '-C', DINO_ROOT, 'checkout', DINO_COMMIT])
ARTIFACTS.mkdir(parents=True, exist_ok=True)
ENCODER, CHECKPOINT = ARTIFACTS / 'encoder.pth.tar', ARTIFACTS / 'fixed-best.pt'
for source, destination in ((ENCODER_DRIVE, ENCODER), (WM_DRIVE, CHECKPOINT)):
    assert source.is_file(), f'Missing frozen artifact: {source}'
    if not destination.exists() or destination.stat().st_size != source.stat().st_size: shutil.copy2(source, destination)
COMMON = ['--parent-dir', str(PARENT), '--clean-dir', str(CLEAN), '--adjacent-dir', str(ADJACENT), '--output-dir', str(OUTPUT),
          '--world-model-checkpoint', str(CHECKPOINT), '--encoder-checkpoint', str(ENCODER),
          '--dino-wm-root', str(DINO_ROOT), '--device', 'cuda', '--precision', 'fp32', '--encoder-batch-size', '8']
def stage(name, recipe=None, band_index=None):
    extra = ['--recipe', recipe] if recipe else []
    if band_index is not None: extra += ['--band-index', str(band_index)]
    if name == 'prepare': extra += ['--without-anchor']
    if name == 'compare': extra += ['--baseline-dir', str(BASELINE)]
    started = time.monotonic()
    shell([sys.executable, '-m', 'src.mylonas_ijepa_wm.curriculum_multiband', name, *COMMON, *extra],
          cwd=REPO)
    print(f"{name} {recipe or ''} {band_index or ''} took {(time.monotonic() - started) / 60:.1f} minutes")
print('GPU:', torch.cuda.get_device_name())
print('Torch:', torch.__version__)
print('Parent:', PARENT)
print('Clean adapter source:', CLEAN)
print('Adjacent evaluation source:', ADJACENT)
print('New output:', OUTPUT)
print('Historical anchored reference:', BASELINE)


## Prepare the ranking-only experiment

The manifest fixes exactly the same 20 evaluation cases per band and 40 separate collection starts per new band. The only active loss is the existing softplus temporal-ranking objective. Its raw replay median is retained as a normalization scale; there is no penalty pulling costs toward raw MSE.


In [ ]:
stage('prepare')
manifest = json.loads((OUTPUT / 'manifest.json').read_text())
assert manifest['fitting']['anchor_weight'] == 0
print('Fitting:', manifest['fitting'])
print('Initial adapters: identity → near-band ranking-only fit')
print('Bands:', manifest['bounds'])
print('Evaluation cases:', {key: len(rows) for key, rows in manifest['evaluation'].items()})
print('Collection attempts per curriculum:', {key: len(rows) for key, rows in manifest['collection'].items()})
stage('grid')
stage('baseline')


## Train and evaluate both recipes independently

For each recipe: fit its initial adapter on saved near experience, evaluate it, then repeat **evaluate → collect 40 attempts → add successful replay → fit 500 updates → evaluate current/earlier bands**. A band with no usable successful experience stops that recipe. The other recipe still runs.

The plots show ranking loss; anchoring is disabled. Each optimizer update samples a new batch, so individual loss values fluctuate. Lower training loss alone does not establish better planning.

Each immediate heatmap preview uses an expanded range for that checkpoint. After the run, the full comparison plots use one expanded scale across both recipes and all saved versions. Numeric costs are saved without clipping.


In [ ]:
from IPython.display import display, Image
import numpy as np
import matplotlib.pyplot as plt

def show_heatmap(recipe, version):
    folder = OUTPUT / f'branches/{recipe}/heatmaps'
    data = json.loads((folder / f'v{version}.json').read_text())
    points = np.asarray(data['points'])
    xs, ys = np.unique(points[:, 0]), np.unique(points[:, 1])
    raw, adapted = np.asarray(data['raw']), np.asarray(data['adapted'])
    difference = adapted - raw
    limit = max(float(raw.max()), float(adapted.max())) * 1.03
    difference_limit = max(float(np.abs(difference).max()), 1e-12) * 1.03
    fig, axes = plt.subplots(1, 3, figsize=(11, 6), constrained_layout=True)
    for ax, values, title, is_difference in zip(axes, (raw, adapted, difference),
            ('Raw MSE', 'Adapted MSE', 'Adapted minus raw'), (False, False, True)):
        image = ax.pcolormesh(xs, ys, values.reshape(len(ys), len(xs)), shading='nearest',
            cmap='RdBu_r' if is_difference else 'viridis',
            vmin=-difference_limit if is_difference else 0,
            vmax=difference_limit if is_difference else limit)
        for boundary in (8, 14, 22, 30, 36):
            ax.axhline(boundary, color='white', linestyle='--', linewidth=.6, alpha=.7)
        ax.scatter(50, 42, marker='*', c='orange', edgecolors='black', s=80)
        ax.set(title=title, xlabel='x', ylabel='y', aspect='equal')
        fig.colorbar(image, ax=ax, shrink=.7)
    fig.suptitle(f'{recipe} v{version}: ranking-only; full-range preview\n'
                 'Preview rescales per version; final plots share a scale across all versions.')
    plt.show(); plt.close(fig)
    print(f'Raw max={raw.max():.6g}; adapted max={adapted.max():.6g}')

errors = {}
for recipe in ('original', 'clean'):
    print('\n' + '='*70 + '\nRANKING-ONLY CURRICULUM:', recipe)
    try:
        stage('initial', recipe)
        branch = OUTPUT / f'branches/{recipe}'
        if (branch / 'stopped.json').exists():
            print('THIS CURRICULUM STOPPED:', (branch / 'stopped.json').read_text())
            continue
        initial = json.loads((branch / 'updates/v0/training.json').read_text())
        assert initial['initialization'] == 'identity' and initial['anchor_weight'] == 0
        print('Initial fit:', initial['initialization'], '| anchor weight:', initial['anchor_weight'])
        print('Initial ranking loss, first/last:', initial['history'][0]['rank_loss'], initial['history'][-1]['rank_loss'])
        fig, ax = plt.subplots(figsize=(8, 2.6), constrained_layout=True)
        ax.plot([row['update'] for row in initial['history']], [row['rank_loss'] for row in initial['history']])
        ax.set(title=f'{recipe}: initial near-band ranking-only fit', xlabel='Optimizer update', ylabel='Ranking loss')
        plt.show(); plt.close(fig)
        show_heatmap(recipe, 0)
        for band_index in (1, 2, 3):
            stage('band', recipe, band_index)
            if (branch / 'stopped.json').exists():
                print('THIS CURRICULUM STOPPED:', (branch / 'stopped.json').read_text())
                break
            folder = branch / f'updates/v{band_index}'
            status = json.loads((folder / 'status.json').read_text())
            training = json.loads((folder / 'training.json').read_text())
            assert training['anchor_weight'] == 0 and not training['anchoring_enabled']
            print(f"Collection successes: {status['collection_successes']}/{status['collection_attempts']}")
            print(f"Replay: {status['old_episodes']} earlier + {status['new_episodes']} new episodes")
            print('Episodes per usable band:', training['band_episode_counts'])
            print('Sampled pairs per band:', training['band_sample_counts'])
            print('Ranking loss, first/last:', training['history'][0]['rank_loss'], training['history'][-1]['rank_loss'])
            print('Incoming checkpoint:', status['inputs']['incoming_sha256'])
            print('New checkpoint:', status['checkpoint_sha256'])
            show_heatmap(recipe, band_index)
            display(Image(filename=str(folder / 'training_and_experience.png')))
    except subprocess.CalledProcessError as error:
        errors[recipe] = str(error)
        print(f'{recipe} encountered an execution error; continuing with the other recipe.')
print('Execution errors:', errors or 'none')


## Planning results and historical anchored comparison

The new report uses the same evaluation cases across checkpoints. Successes, primitive actions, distances and paired wins/losses against raw and the incoming version remain available. Historical comparisons below read the anchored run without modifying it.

The historical run is a reference: the policies can collect different experience, and the old original near-band fit did not persist its target tokens. Both new initial fits use the retained shared target. Treat observed differences as experiment outcomes rather than attributing every change uniquely to anchoring.


In [ ]:
stage('report')
report = json.loads((OUTPUT / 'report.json').read_text())
bands = ('near', 'band_1', 'band_2', 'band_3')
print('\nRAW REFERENCE')
if report.get('raw_error'): print('Raw results unavailable:', report['raw_error'])
for band in bands:
    row = report['raw'][band]
    print(band, row)
for recipe in ('original', 'clean'):
    branch = report['branches'][recipe]
    print(f"\n{recipe.upper()} — status: {branch['status']}")
    if branch.get('error'): print('Results unavailable:', branch['error'])
    print('Each cell below: successes / primitive actions; N/A = no completed evaluation')
    print('version      near         band 1       band 2       band 3')
    for version, rows in branch['versions'].items():
        cells = ['N/A' if rows[band] is None else f"{rows[band]['successes']}/20 | {rows[band]['primitive_steps']}" for band in bands]
        print(f"v{version:<7}" + ''.join(f'{cell:<14}' for cell in cells))
        print('  Mean final distances:', {band: None if rows[band] is None else round(rows[band]['mean_final_distance'], 3) for band in bands})
    print('Collection:', json.dumps(branch['collections'], indent=2))
    print('Paired changes vs raw and incoming checkpoint:', json.dumps(branch['comparisons'], indent=2))
    print('Primitive-action journal:', json.dumps(branch['budget'], indent=2))
    if branch['stopped']: print('Stopped reason:', branch['stopped'])
print('\nTotal primitive interactions:', report['total_primitive_interactions'])
print('Known primitive interactions:', report['known_primitive_interactions'], '| total is a lower bound/incomplete:', report['total_count_is_lower_bound'])
print('Heatmap rendering primitive interactions:', report['diagnostic_primitive_interactions'])
print('Interpretation:', report['note'])
display(Image(filename=str(OUTPUT / 'curriculum_matrix.png')))
print('Full report:', OUTPUT / 'report.json')

try:
    stage('compare')
    comparison = json.loads((OUTPUT / 'anchor_comparison.json').read_text())
    print('\nHISTORICAL ANCHORED → WITHOUT ANCHOR (successes / actions)')
    for recipe, branch in comparison['branches'].items():
        print('\n', recipe.upper())
        if branch['error']: print('Comparison unavailable:', branch['error'])
        for version, rows in branch['versions'].items():
            print('Version', version)
            for band, cell in rows.items():
                if cell is None:
                    print(' ', band, 'N/A — one or both evaluations unavailable')
                else:
                    a, b, paired = cell['anchored'], cell['without_anchor'], cell['paired']
                    print(f"  {band}: {a['successes']}/20 | {a['primitive_steps']} → "
                          f"{b['successes']}/20 | {b['primitive_steps']}; "
                          f"paired wins={paired['wins']}, losses={paired['losses']}")
    print(comparison['note'])
except subprocess.CalledProcessError as error:
    print('Historical comparison could not be validated:', error)
    print('The new report and full-range diagnostic plots remain available.')


## Full-range heatmaps and vertical profiles

All plots below use a **single expanded scale across both recipes and every available version**. This exposes cost growth without saturating at the old raw-cost ceiling. The vertical profiles hold x fixed and move toward goal y=42 from left to right: a falling curve assigns lower cost to that progress; a rising curve assigns higher cost.

Profiles are costs of real rendered images, not the world model's predicted action sequences. No additional simulation or training is performed in this section.


In [ ]:

import hashlib
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT = OUTPUT
SAVE_FIGURES = True
FIGURES = ROOT / 'diagnostics' / 'cost_profiles'
RECIPES = ('original', 'clean')
BANDS = ((8, 14, 'band 3'), (14, 22, 'band 2'), (22, 30, 'band 1'), (30, 36, 'near'))
GOAL = (50, 42)

# Read the exact saved numbers, with no encoder, world model or adapter loading.
maps, rows = {}, []
reference_points = reference_raw = reference_signature = None
for recipe in RECIPES:
    maps[recipe] = {}
    for version in range(4):
        path = ROOT / f'branches/{recipe}/heatmaps/v{version}.json'
        if not path.exists():
            print(f'{recipe} v{version}: no saved map; omitted')
            continue
        commit_path = path.with_name(f'v{version}.commit.json')
        if not commit_path.exists():
            raise ValueError(f'Missing completed-map record: {commit_path}')
        commit = json.loads(commit_path.read_text())
        payload = path.read_bytes()
        if hashlib.sha256(payload).hexdigest() != commit['data_sha256']:
            raise ValueError(f'Saved numeric map hash differs: {path}')
        data = json.loads(payload)
        if data['recipe'] != recipe or data['version'] != version:
            raise ValueError(f'Wrong recipe/version: {path}')
        if any(data.get(key) != value for key, value in commit['inputs'].items()):
            raise ValueError(f'Map provenance differs: {path}')
        points = np.asarray(data['points'], dtype=float)
        raw = np.asarray(data['raw'], dtype=float)
        adapted = np.asarray(data['adapted'], dtype=float)
        if (points.ndim != 2 or points.shape[1] != 2 or raw.shape != (len(points),)
                or adapted.shape != raw.shape or not np.isfinite(points).all()
                or not np.isfinite(raw).all() or not np.isfinite(adapted).all()
                or (raw < 0).any() or (adapted < 0).any()):
            raise ValueError(f'Invalid saved numeric map: {path}')
        if reference_points is None:
            reference_points, reference_raw, reference_signature = points, raw, data['signature']
        elif (not np.array_equal(points, reference_points)
              or not np.array_equal(raw, reference_raw) or data['signature'] != reference_signature):
            raise ValueError(f'Maps do not share the same grid/raw costs/goal/runtime: {path}')
        maps[recipe][version] = adapted
        rows.append(dict(recipe=recipe, version=f'v{version}',
            raw_max=float(raw.max()), adapted_max=float(adapted.max()),
            previous_display_max=data['display_max'],
            above_previous_scale=int((adapted > data['display_max']).sum())))
if reference_points is None:
    raise FileNotFoundError(f'No saved heatmap JSON files found under {ROOT}')
xs = np.unique(reference_points[:, 0])
ys = np.unique(reference_points[:, 1])
expected = np.array([(x, y) for y in ys for x in xs])
if not np.array_equal(expected, reference_points):
    raise ValueError('Expected the complete saved rectangular grid, ordered by y then x')
RAW = reference_raw.reshape(len(ys), len(xs))
for recipe in RECIPES:
    maps[recipe] = {v: a.reshape(RAW.shape) for v, a in maps[recipe].items()}
maximum = max([float(RAW.max())] + [float(a.max()) for versions in maps.values() for a in versions.values()])
DISPLAY_MAX = maximum * 1.03
print(f'Expanded shared color/cost limit: {DISPLAY_MAX:.6f}; maximum saved cost: {maximum:.6f}')
print('v0 = freshly fitted on near experience; v1 = after band 1; v2 = after band 2; v3 = after band 3, if saved.')
print(f"{'recipe':<10} {'version':<8} {'raw max':>10} {'adapted max':>13} {'old ceiling':>13} {'above old ceiling':>19}")
for row in rows:
    print(f"{row['recipe']:<10} {row['version']:<8} {row['raw_max']:>10.6f} "
          f"{row['adapted_max']:>13.6f} {row['previous_display_max']:>13.6f} "
          f"{row['above_previous_scale']:>15}/{len(reference_points)}")

def finish(figure, name):
    if SAVE_FIGURES:
        FIGURES.mkdir(parents=True, exist_ok=True)
        figure.savefig(FIGURES / f'{name}.png', dpi=170, bbox_inches='tight')
        figure.savefig(FIGURES / f'{name}.pdf', bbox_inches='tight')
    plt.show()
    plt.close(figure)


In [ ]:
for recipe in RECIPES:
    if not maps[recipe]:
        continue
    panels = [('Raw', RAW)] + [(f'Adapter v{v}', a) for v, a in sorted(maps[recipe].items())]
    fig, axes = plt.subplots(1, len(panels), figsize=(3.1 * len(panels), 6.2),
                             sharex=True, sharey=True, squeeze=False, constrained_layout=True)
    for ax, (label, values) in zip(axes[0], panels):
        image = ax.pcolormesh(xs, ys, values, shading='nearest', cmap='viridis',
                             vmin=0, vmax=DISPLAY_MAX)
        for lo, hi, band in BANDS:
            ax.axhline(lo, color='white', linestyle='--', linewidth=0.7, alpha=0.7)
        ax.axhline(36, color='white', linestyle='--', linewidth=0.7, alpha=0.7)
        ax.scatter(*GOAL, marker='*', s=100, c='orange', edgecolors='black', linewidths=0.6)
        ax.set(title=f'{label}\nmax = {values.max():.4f}', xlabel='Position x (simulation units)',
               aspect='equal', xlim=(xs.min()-1, xs.max()+1), ylim=(ys.min()-1, ys.max()+1))
    axes[0, 0].set_ylabel('Position y (simulation units)')
    fig.colorbar(image, ax=axes[0].tolist(), shrink=0.8, label='Latent MSE — same expanded scale for every map')
    fig.suptitle(f'{recipe}: full saved cost range — no color saturation', fontsize=13)
    finish(fig, f'{recipe}_unclipped_maps')


In [ ]:
# Each curve holds x fixed and moves vertically through real grid observations.
# Stop at goal y: beyond y=42, increasing y moves away from the goal again.
requested_x = (40, 46, 50, 54)
x_indices = [int(np.argmin(np.abs(xs-x))) for x in requested_x]
if len(set(x_indices)) != len(x_indices):
    raise ValueError('Saved grid cannot supply four distinct requested x profiles')
keep = ys <= GOAL[1]
profile_y = ys[keep]
styles = {0: ('#0072b2', '-'), 1: ('#d55e00', '--'),
          2: ('#009e73', '-.'), 3: ('#cc79a7', ':')}
for recipe in RECIPES:
    if not maps[recipe]:
        continue
    fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey=True,
                             constrained_layout=True)
    for ax, column in zip(axes.flat, x_indices):
        for index, (lo, hi, band) in enumerate(BANDS):
            ax.axvspan(lo, hi, color='0.5', alpha=0.10 if index % 2 == 0 else 0.025)
            ax.axvline(lo, color='0.6', linestyle=':', linewidth=0.7)
            ax.text((lo+hi)/2, DISPLAY_MAX * 0.97, band, ha='center', va='top', fontsize=9)
        ax.plot(profile_y, RAW[keep, column], color='black', linewidth=2,
                marker='o', markersize=3, label='Raw')
        for version, values in sorted(maps[recipe].items()):
            color, linestyle = styles[version]
            ax.plot(profile_y, values[keep, column], color=color, linestyle=linestyle,
                    marker='.', markersize=5, linewidth=1.7, label=f'Adapter v{version}')
        ax.axvline(GOAL[1], color='0.4', linestyle='--', linewidth=1)
        ax.set(title=f'Fixed x = {xs[column]:g}', xlim=(profile_y.min(), GOAL[1]),
               ylim=(0, DISPLAY_MAX), xlabel='Position y → toward goal y=42', ylabel='Latent MSE')
        ax.grid(axis='y', alpha=0.18)
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='outside lower center', ncol=len(labels), frameon=False)
    fig.suptitle(f'{recipe}: cost along vertical paths\n'
                 'Read left → right: falling = lower cost toward goal; rising = higher cost toward goal', fontsize=12)
    finish(fig, f'{recipe}_vertical_profiles')
if SAVE_FIGURES:
    print('Saved PNG and PDF figures:', FIGURES)


## Return the results

Download this executed notebook (**File → Download → Download .ipynb**) and attach it here. We will compare planning outcomes, cost magnitudes, the lower-band hill, and ranking-loss trends against the historical anchored run. The new numeric maps and profile PNG/PDF files are saved in `multiband_no_anchor_v1`.
